# Leakage-safe Email Spam Experiment

This is the recommended notebook. Run `python -m spam_classifier.train --data emails.csv --out artifacts` to create full CV reports and figures. The original `email-spam-detection.ipynb` is retained as a historical notebook; it fitted TF-IDF **before** splitting, causing information leakage.


In [ ]:
from spam_classifier.data import load_dataset
from spam_classifier.models import build_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix
df, stats = load_dataset("emails.csv")
stats

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=df["spam"], random_state=42)
X_train, y_train = df.loc[train_idx,"text"], df.loc[train_idx,"spam"]
X_test, y_test = df.loc[test_idx,"text"], df.loc[test_idx,"spam"]
print(len(train_idx), len(test_idx))

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
pipe = build_pipeline("multinomial_nb")
scores = cross_validate(pipe, X_train, y_train, scoring=["f1", "precision", "recall"], cv=cv)
{metric: scores[metric].mean() for metric in ("test_f1", "test_precision", "test_recall")}

In [ ]:
pipe.fit(X_train, y_train)
predictions = pipe.predict(X_test)
print(classification_report(y_test, predictions, target_names=["ham", "spam"]))
print(confusion_matrix(y_test, predictions, labels=[0,1]))